# EduHinglish: Model Server on Colab T4 (for the backend)

Runs the trained model and exposes it as an HTTP API with a public URL. The project backend (`backend/`) calls this URL.

```
Frontend ──► backend/ (FastAPI, any machine) ──HTTPS + X-API-Key──► this notebook (T4 GPU)
                                                                    M1 intent → M2 ChromaDB → M3 Gemma + LoRA
```

**Secrets** (🔑 sidebar, enable notebook access):
| Name | What |
|---|---|
| `HF_TOKEN` | Hugging Face token (read is enough) |
| `MODEL_API_KEY` | any long random string; put the same value in `backend/.env` |
| `NGROK_AUTHTOKEN` | optional: gives a **fixed URL** (see the Tunnel cell) |
| `GITHUB_TOKEN` | only if the repo is private |

> **Runtime**: Runtime → Change runtime type → **T4 GPU**. Run the pip cell, **Restart session**, then **Run all**.

In [ ]:
!pip install -q -U "bitsandbytes>=0.43.0" "transformers>=4.47.0" "peft>=0.14.0" "accelerate>=1.0.0" \
    "chromadb>=0.4.24" "sentence-transformers>=2.7.0" nltk colorama \
    "fastapi>=0.111" "uvicorn>=0.29" pyngrok

## 1. Settings

In [ ]:
GITHUB_REPO = "https://github.com/PersuasivePost/EduHinglish.git"
BRANCH      = "develop"

HF_ADAPTER_REPO = "your-hf-username/eduhinglish-gemma-3-4b-it-lora"   # Gemma LoRA (notebook 05)
HF_INTENT_REPO  = "your-hf-username/eduhinglish-muril-intent"          # None = no intent hint

BASE_DIR   = "/content/drive/MyDrive/EduHinglish"
KB_CACHE   = f"{BASE_DIR}/knowledge_base_v4"     # same cache as notebook 06
REBUILD_KB = False

PORT = 8000
# Fixed URL with ngrok (free): create a static domain at https://dashboard.ngrok.com/domains and
# paste it here, e.g. "eduhinglish-demo.ngrok-free.app". Leave "" for a random ngrok URL.
# Without the NGROK_AUTHTOKEN secret, a Cloudflare quick tunnel is used (random URL, no account).
NGROK_DOMAIN = ""

## 2. Code, secrets, Drive

In [ ]:
import os, sys, shutil, subprocess
from google.colab import drive, userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.environ["MODEL_API_KEY"] = secret("MODEL_API_KEY") or ""
if not os.environ["MODEL_API_KEY"]:
    print("WARNING: no MODEL_API_KEY secret, so anyone with the URL can use the GPU.")

gh = secret("GITHUB_TOKEN")
url = GITHUB_REPO.replace("https://", f"https://{gh}@") if gh else GITHUB_REPO
REPO = "/content/EduHinglish"
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
subprocess.run([sys.executable, "scripts/download_nltk_data.py"], check=True)

## 3. Knowledge base (restored from Drive; built once)

In [ ]:
KB_LOCAL = f"{REPO}/src/knowledge_base"

def kb_count(path):
    from retriever import NCERTRetriever
    try:
        return NCERTRetriever(db_path=path).get_collection_stats()["total_documents"]
    except Exception as e:
        print("KB load failed:", e)
        return 0

if not REBUILD_KB and os.path.isdir(KB_CACHE):
    shutil.rmtree(KB_LOCAL, ignore_errors=True)
    shutil.copytree(KB_CACHE, KB_LOCAL)
if kb_count(KB_LOCAL) == 0:
    subprocess.run([sys.executable, "scripts/build_knowledge_base.py", "--force"], check=True)
    shutil.rmtree(KB_CACHE, ignore_errors=True)
    shutil.copytree(KB_LOCAL, KB_CACHE)
print("Knowledge base chunks:", kb_count(KB_LOCAL))

## 4. Load the model (~3–5 min)

In [ ]:
os.environ["EDUHINGLISH_ADAPTER"] = HF_ADAPTER_REPO
if HF_INTENT_REPO:
    os.environ["EDUHINGLISH_INTENT_MODEL"] = HF_INTENT_REPO
os.environ["EDUHINGLISH_KB_PATH"] = KB_LOCAL

from pipeline import EduHinglishPipeline
pipe = EduHinglishPipeline(mode="full", use_intent_model=bool(HF_INTENT_REPO))
print("LoRA loaded:", pipe.generator.lora_loaded, "|", pipe.generator.adapter_source)

## 5. Start the API server

In [ ]:
import model_server, requests
server = model_server.start_in_thread(pipe, port=PORT)
print(requests.get(f"http://localhost:{PORT}/health").json())

## 6. Public URL (tunnel)

In [ ]:
import re, time

ngrok_token = secret("NGROK_AUTHTOKEN")
if ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    kwargs = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
    PUBLIC_URL = ngrok.connect(PORT, "http", **kwargs).public_url
else:
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                       check=True)
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    log = open("/content/cloudflared.log", "w")
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                              stdout=log, stderr=subprocess.STDOUT)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m:
            PUBLIC_URL = m.group(0)
            break
        time.sleep(1)
    assert PUBLIC_URL, "cloudflared did not print a URL; see /content/cloudflared.log"

# Save it on Drive so teammates can find the current URL
with open(f"{BASE_DIR}/model_server_url.txt", "w") as f:
    f.write(PUBLIC_URL + "\n")

print("=" * 70)
print("MODEL_SERVER_URL =", PUBLIC_URL)
print("Put this in backend/.env (with MODEL_API_KEY) and restart the backend.")
print("=" * 70)

## 7. Test through the public URL (same call the backend makes)

In [ ]:
headers = {"X-API-Key": os.environ["MODEL_API_KEY"], "ngrok-skip-browser-warning": "1"}
print(requests.get(f"{PUBLIC_URL}/health", headers=headers, timeout=30).json())
r = requests.post(f"{PUBLIC_URL}/ask", headers=headers, timeout=180,
                  json={"query": "Ohm ka law kya hai?", "subject": "physics", "class_num": "10"})
print(r.status_code)
res = r.json()
print("ANSWER:", res.get("answer"))
print("INTENT:", res.get("intent"), res.get("intent_confidence"))
print("TIMINGS:", res.get("timings"))

## 8. Keep the server running
The server runs as long as this runtime is alive. This cell prints a heartbeat every 5 minutes.
Stop it (■) when the demo is over; that also shuts down the URL.

In [ ]:
while True:
    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=10).json()
        print(time.strftime("%H:%M:%S"), "ok | busy:", h.get("busy"), "|", PUBLIC_URL)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "health check failed:", e)
    time.sleep(300)